In [0]:
%run "../../utils/00_config"

In [0]:
"../../utils/02_data_quality"

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Silver sources
fact_orders = spark.table(
    get_table("intermediate", "fact_order_observation")
)

# Source validation
print(f"Fact order observations: {fact_orders.count():,}")

# display(fact_orders.limit(5))

In [0]:
# Player status and offer visibility distribution
'''
display(
    fact_orders
    .groupBy("player_status", "visible")
    .agg(
        F.count("*").alias("total_orders"),
        F.countDistinct("order_id").alias("unique_orders"),
        F.min("platinum").alias("min_price"),
        F.max("platinum").alias("max_price")
    )
    .orderBy("player_status", "visible")
)
'''


In [0]:

# Filter active and visible market offers
'''
WTS business_rule eligible filters
- The player is online or in-game.
- The offer is visible.
- The price is not zero.
- The price is greater than zero.
'''
eligible_orders = (
    fact_orders
    .filter(
        F.col("player_status").isin("online", "ingame")
        & (F.col("visible") == True)
        & F.col("platinum").isNotNull()
        & (F.col("platinum") > 0)
    )
)

print(f"Eligible orders: {eligible_orders.count():,}")

'''display(
    eligible_orders
    .groupBy("product_slug", "item_slug", "order_type")
    .agg(
        F.count("*").alias("total_orders"),
        F.min("platinum").alias("min_price"),
        F.max("platinum").alias("max_price")
    )
    .orderBy("product_slug", "item_slug", "order_type")
)'''


In [0]:
# Calculate price distribution by item and order type

MIN_SAMPLE_SIZE = 8
PERCENTILE_ACCURACY = 10000

# Each item has independent price limits for WTB and WTS,
# calculated separately for each execution.
# Thresholds are inclusive: prices equal to either limit
# are not considered outliers.

price_stats = (
    eligible_orders
    .groupBy("execution_id", "item_slug", "order_type")
    .agg(
        F.count("*").alias("total_orders"),
        F.percentile_approx(
            "platinum", 0.25, PERCENTILE_ACCURACY
        ).alias("q1"),
        F.percentile_approx(
            "platinum", 0.75, PERCENTILE_ACCURACY
        ).alias("q3"),
        F.min("platinum").alias("min_price"),
        F.max("platinum").alias("max_price")
    )
    .withColumn("iqr", F.col("q3") - F.col("q1"))
    .withColumn("lower_bound", F.col("q1") - 1.5 * F.col("iqr"))
    .withColumn("upper_bound", F.col("q3") + 1.5 * F.col("iqr"))
    .withColumn(
        "quality_status",
        F.when(
            F.col("total_orders") < MIN_SAMPLE_SIZE,
            "insufficient_sample"
        ).otherwise("evaluated")
    )
)

# display(price_stats.orderBy("item_slug", "order_type"))

In [0]:
# Attach price statistics to individual orders
quality_orders = (
    eligible_orders
    .join(
        price_stats,
        on=["execution_id", "item_slug", "order_type"],
        how="left"
    )
    .withColumn(
        "is_price_outlier",
        F.when(
            F.col("quality_status") == "insufficient_sample",
            F.lit(None).cast("boolean")
        ).otherwise(
            (F.col("platinum") < F.col("lower_bound"))
            | (F.col("platinum") > F.col("upper_bound"))
        )
    )
    .withColumn(
        "price_quality_status",
        F.when(
            F.col("quality_status") == "insufficient_sample",
            "insufficient_sample"
        )
        .when(F.col("is_price_outlier") == True, "outlier")
        .when(F.col("is_price_outlier") == False, "within_iqr")
        .otherwise("unclassified")
    )
)

# Validate classification
'''display(
    quality_orders
    .groupBy("item_slug", "order_type", "price_quality_status")
    .agg(
        F.count("*").alias("total_orders"),
        F.min("platinum").alias("min_price"),
        F.max("platinum").alias("max_price")
    )
    .orderBy("item_slug", "order_type", "price_quality_status")
)'''

In [0]:

# Validate price quality classification

required_fields = [
    "execution_id",
    "order_id",
    "item_slug",
    "order_type",
    "platinum",
    "quality_status",
    "price_quality_status"
]

# Required fields validation
null_conditions = [
    F.col(column).isNull()
    for column in required_fields
]

null_count = (
    quality_orders
    .filter(F.expr(" OR ".join(
        f"`{column}` IS NULL" for column in required_fields
    )))
    .count()
)

assert null_count == 0, (
    f"Found {null_count} records with null required fields."
)

# Unique key validation
duplicate_count = (
    quality_orders
    .groupBy("execution_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_count == 0, (
    f"Found {duplicate_count} duplicate orders."
)

# Record count validation
eligible_count = eligible_orders.count()
classified_count = quality_orders.count()

assert eligible_count == classified_count, (
    f"Record count mismatch: eligible={eligible_count}, "
    f"classified={classified_count}"
)

# Classification validation
valid_statuses = [
    "insufficient_sample",
    "outlier",
    "within_iqr"
]

invalid_classifications = (
    quality_orders
    .filter(
        F.col("price_quality_status").isNull()
        | ~F.col("price_quality_status").isin(valid_statuses)
    )
    .count()
)

assert invalid_classifications == 0, (
    f"Found {invalid_classifications} invalid classifications."
)

print(f"Eligible orders: {eligible_count:,}")
print(f"Classified orders: {classified_count:,}")
print("Required fields validation passed.")
print("Unique keys validation passed.")
print("Classification validation passed.")


In [0]:
# Materialize Gold intermediate table
from delta.tables import DeltaTable

# MERGE incremental
'''
- if the table does not exist, create it.
- if it exists, update records with the same key (execution_id, order_id).
- insert new records.
- prevents duplicates during reprocessing, provided the source does not have duplicate keys.
'''

GOLD_TARGET = get_gold_table(
    "intermediate",
    "int_item_price_quality"
)

merge_condition = """
    target.execution_id = source.execution_id
    AND target.order_id = source.order_id
"""

if spark.catalog.tableExists(GOLD_TARGET):
    (
        DeltaTable.forName(spark, GOLD_TARGET)
        .alias("target")
        .merge(
            quality_orders.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        quality_orders.write
        .format("delta")
        .mode("append")
        .saveAsTable(GOLD_TARGET)
    )

print(f"Table materialized successfully: {GOLD_TARGET}")

In [0]:
# Validate materialized table

gold_quality = spark.table(GOLD_TARGET)

expected_count = quality_orders.count()
actual_count = gold_quality.count()

assert expected_count == actual_count, (
    f"Materialization mismatch: expected={expected_count}, "
    f"actual={actual_count}"
)

duplicate_count = (
    gold_quality
    .groupBy("execution_id", "order_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

assert duplicate_count == 0, (
    f"Found {duplicate_count} duplicate orders."
)

print(f"Expected records: {expected_count:,}")
print(f"Materialized records: {actual_count:,}")
print(f"Duplicate keys: {duplicate_count}")
print("Gold materialization validation passed.")